In [ ]:
dbutils.widgets.dropdown("catalog", "madrid_dev", ["dev", "pro", "madrid_dev"])
dbutils.widgets.text("confirm", "", "Type catalog name to confirm DROP")
CATALOG_TARGET = dbutils.widgets.get("catalog")
CONFIRM = dbutils.widgets.get("confirm")

In [ ]:
# Skip protected schemas
schemas_to_drop = [
    row["databaseName"]
    for row in spark.sql(f"SHOW SCHEMAS IN {CATALOG_TARGET}").collect()
    if row["databaseName"] != "information_schema"
]
print(f"Catalog: {CATALOG_TARGET}")
print(f"Schemas to drop: {schemas_to_drop}")

In [ ]:
CONFIRM = dbutils.widgets.get("confirm")
CATALOG_TARGET = dbutils.widgets.get("catalog")

if CONFIRM == CATALOG_TARGET:
    for schema in schemas_to_drop:
        spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG_TARGET}.{schema} CASCADE")
        print(f"Dropped schema: {CATALOG_TARGET}.{schema}")
    spark.sql(f"DROP CATALOG IF EXISTS {CATALOG_TARGET} CASCADE")
    print(f"Dropped catalog: {CATALOG_TARGET}")
else:
    print("DRY RUN - confirm mismatch, nothing dropped")
    print(f"Would drop catalog '{CATALOG_TARGET}' and schemas: {schemas_to_drop}")

In [ ]:
remaining = [row["catalog"] for row in spark.sql("SHOW CATALOGS").collect()]
if CATALOG_TARGET in remaining:
    print(f"'{CATALOG_TARGET}' still exists")
else:
    print(f"'{CATALOG_TARGET}' dropped")

In [ ]:
# Delete dev.ml model, function, endpoint, then tables/volumes
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.registered_models.delete(full_name="dev.ml.aire_forecast")
w.functions.delete(name="dev.ml.aire_forecast")
w.serving_endpoints.delete(name="aireforecast")
spark.sql("DROP SCHEMA IF EXISTS dev.ml CASCADE")